#### Author's Information:
        Author : Ali Rizvi.
        Stu ID : 2410057
        Module : HEP502 of MSc Computer Sciences with Artificial Intelligence.
        Title  : Comparative Evaluation of MLP and CNN on Fashion-MNIST.
        Purpose: Final Assessment
---

| #| ## of Notebook | Title of Notebook |
|--|---|---|
| 1st| 00 | Data Loading and Sanity Checks |
| 2| 01 | Baseline MLP |
| 3| 02 | Baseline CNN |
| **4**| **03** | **Regularisation Experiments** |
| 5| 04 | Depth and Capacity Variance |
| 6| 05 | Robustness Testing |
| 7| 06 | Consolidated Analysis |

**This is Notebook 03,** 4th of this 7-notebook series. The following are the goals of this notebook:
## Goals:
Compare regularisation strategies for:
- MLP
- CNN
Regularisation:
1) Dropout
2) L2 weight decay
3) Optional Dropout + L2

Training settings like the following are fixed for fair comparison between both models
- Model
- Plots
- Metrics JSON per run

**This notebook contains experimentations with Regularization techniques**

#### Importing libraries

In [31]:
import json
import time
from dataclasses import dataclass
from pathlib import Path
import csv

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

#### Reproductibility and Paths

In [18]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [19]:
OUTPUT_DIR = Path("outputs")
FIG_DIR = OUTPUT_DIR / "figures"
LOG_DIR = OUTPUT_DIR / "logs"
MODEL_DIR = OUTPUT_DIR / "models"

directories = [OUTPUT_DIR, FIG_DIR, LOG_DIR, MODEL_DIR]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)
    
print(f"Outputs will be saved to: {OUTPUT_DIR.resolve()}")

Outputs will be saved to: C:\Users\ALI\PG\HEP502_work\alirizvi_2410057_hep502\code_mlp_vs_cnn\outputs


#### Load prepared data

In [20]:
data_path = OUTPUT_DIR / "fashion_mnist_prepared_00.npz"

assert data_path.exists(), f"Missing file: {data_path}. Run notebook 00 first"

data = np.load(data_path)
x_train = data["x_train"]
y_train = data["y_train"]
x_val = data["x_val"]
y_val = data["y_val"]
x_test = data["x_test"]
y_test = data["y_test"]

print(f"x_train shape: {x_train.shape} | x_train type: {x_train.dtype}")
print(f"x_val shape: {x_val.shape} | x_val type: {x_val.dtype}")
print(f"x_test shape: {x_test.shape} | x_test type: {x_test.dtype}")
print(f"y_train shape: {y_train.shape} | y_train type: {y_train.dtype}")

x_train shape: (48000, 28, 28) | x_train type: float32
x_val shape: (12000, 28, 28) | x_val type: float32
x_test shape: (10000, 28, 28) | x_test type: float32
y_train shape: (48000,) | y_train type: uint8


### Prepare MLP and CNN inputs

In [21]:
def flatten_images(x):
    # convert (N, 28, 28) to (N, 784) for MLP input
    return x.reshape((x.shape[0], -1))

def add_channel_dim(x):
    # Add explicit channel axis. (N, 28, 28) becomes (N, 28, 28, 1) for CNN input
    return x[:, :, :, np.newaxis]

x_train_mlp = flatten_images(x_train)
x_val_mlp = flatten_images(x_val)
x_test_mlp = flatten_images(x_test)

x_train_cnn = add_channel_dim(x_train)
x_val_cnn = add_channel_dim(x_val)
x_test_cnn = add_channel_dim(x_test)

print("MLP input:", x_train_mlp.shape)
print("CNN input:", x_train_cnn.shape)

print(f"MLP Inputs: {x_train_mlp.shape}")
print(f"CNN Inputs: {x_train_cnn.shape}")

MLP input: (48000, 784)
CNN input: (48000, 28, 28, 1)
MLP Inputs: (48000, 784)
CNN Inputs: (48000, 28, 28, 1)


### Plot saving helper functions (same as before)

In [22]:
def save_history_plots(history, out_prefix, fig_dir):
    hist = history.history

    # Loss
    plt.figure(figsize=(6, 4))
    plt.plot(hist["loss"], label="train")
    plt.plot(hist["val_loss"], label="val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(fig_dir / f"{out_prefix}_loss.png", dpi=150)
    plt.close()

    # Accuracy
    plt.figure(figsize=(6, 4))
    plt.plot(hist["accuracy"], label="train")
    plt.plot(hist["val_accuracy"], label="val")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(fig_dir / f"{out_prefix}_accuracy.png", dpi=150)
    plt.close()

### Run configuration
Lets use a small config object so this can be looped cleanly

In [23]:
@dataclass(frozen=True)
class RunConfig:
    # configuration container for a single experiment run
    arch: str # "mlp" or "cnn"
    dropout_rate: float = 0.0 # 0.0 means no dropout
    l2_strength: float = 0.0 # 0.0 means no L2
    hidden_units: int = 128 # baseline units, default for MLP
    conv1_filters: int = 32 # default 1st conv layer
    conv2_filters: int = 64 # default 2nd conv layer

    def run_name(self):
        parts = [self.arch]
        if self.dropout_rate > 0:
            parts.append(f"drop{self.dropout_rate:g}")
        if self.l2_strength > 0:
            parts.append(f"l2{self.l2_strength:g}")
        if len(parts) == 1:
            parts.append("baseline")
        return "_".join(parts)

### Model builders with Dropout and L2 hooks

#### MLP builder

In [24]:
def build_mlp(config: RunConfig, input_shape=(784,), num_classes=10):
    # Apply L2 only if specified in config
    l2_reg = regularizers.l2(config.l2_strength) if config.l2_strength > 0 else None

    inputs = keras.Input(shape=input_shape, name="images")
    x = layers.Dense(
        config.hidden_units,
        activation="relu",
        kernel_regularizer=l2_reg,
        name="dense_1"
    )(inputs)

    # Optionally insert dropout for regularisation
    if config.dropout_rate > 0:
        x = layers.Dropout(config.dropout_rate, name="dropout_1")(x)

    outputs = layers.Dense(
        num_classes,
        activation="softmax",
        kernel_regularizer=l2_reg,
        name="preds"
    )(x)

    model = keras.Model(inputs, outputs, name=config.run_name())
    return model

#### CNN builder

In [27]:
def build_cnn(config: RunConfig, input_shape=(28, 28, 1), num_classes=10):
    # Enable L2 regularisation only when specified in the config
    l2_reg = regularizers.l2(config.l2_strength) if config.l2_strength > 0 else None

    inputs = keras.Input(shape=input_shape, name="images")

    # 1st conv block
    x = layers.Conv2D(
        config.conv1_filters, 3,
        activation="relu",
        padding="same",
        kernel_regularizer=l2_reg,
        name="conv1"
    )(inputs)
    x = layers.MaxPooling2D(2, name="maxpool_1")(x)

    # 2nd conv block with increased depth
    x = layers.Conv2D(
        config.conv2_filters, 3,
        activation="relu",
        padding="same",
        kernel_regularizer=l2_reg,
        name="conv2"
    )(x)
    x = layers.MaxPooling2D(2, name="maxpool_2")(x)

    x = layers.Flatten(name="flatten")(x)  # move from feature maps to dense layers
    x = layers.Dense(
        config.hidden_units,
        activation="relu",
        kernel_regularizer=l2_reg,
        name="dense_1"
    )(x)

    # Optional dropout for additional regularisation
    if config.dropout_rate > 0:
        x = layers.Dropout(config.dropout_rate, name="dropout_1")(x)

    outputs = layers.Dense(
        num_classes,
        activation="softmax",
        kernel_regularizer=l2_reg,
        name="preds"
    )(x)

    model = keras.Model(inputs, outputs, name=config.run_name())
    return model

### Training and Evaluation function

In [28]:
def train_and_evaluate(config: RunConfig):
    # Pick the right preprocessed arrays + model builder based on arch
    if config.arch == "mlp":
        x_tr, x_va, x_te = x_train_mlp, x_val_mlp, x_test_mlp
        model = build_mlp(config, input_shape=(784,), num_classes=10)
    elif config.arch == "cnn":
        x_tr, x_va, x_te = x_train_cnn, x_val_cnn, x_test_cnn
        model = build_cnn(config, input_shape=(28, 28, 1), num_classes=10)
    else:
        raise ValueError(f"Unknown arch: {config.arch}")

    # Keep optimiser/loss consistent across runs
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    EPOCHS = 20
    BATCH_SIZE = 128

    # Track time the full training loop (useful when comparing MLP vs CNN cost)
    start_time = time.time()
    history = model.fit(
        x_tr, y_train,
        validation_data=(x_va, y_val),  # track generalisation during training
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0
    )
    train_time_sec = time.time() - start_time

    # Final held-out evaluation on test set
    test_loss, test_acc = model.evaluate(x_te, y_test, verbose=0)

    num_params = model.count_params()

    # Quick overfitting signals
    final_train_acc = float(history.history["accuracy"][-1])
    final_val_acc   = float(history.history["val_accuracy"][-1])
    generalisation_gap = final_train_acc - final_val_acc

    # Pack everything needed for later
    run_info = {
        "run_name": config.run_name(),
        "arch": config.arch,
        "dropout_rate": float(config.dropout_rate),
        "l2_strength": float(config.l2_strength),
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": 1e-3,
        "train_time_sec": float(train_time_sec),
        "num_params": int(num_params),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "final_train_accuracy": final_train_acc,
        "final_val_accuracy": final_val_acc,
        "generalisation_gap": float(generalisation_gap),
        "acc_per_10k_params": float(test_acc / (num_params / 10_000))  # rough efficiency metric
    }

    return model, history, run_info

#### Experimentation Grid

In [29]:
runs = []

# grid of experimental settings: baseline, +dropout, +L2, +both
experiment_configs = [
    # MLP variants
    RunConfig(arch="mlp", dropout_rate=0.0, l2_strength=0.0),
    RunConfig(arch="mlp", dropout_rate=0.3, l2_strength=0.0),
    RunConfig(arch="mlp", dropout_rate=0.0, l2_strength=1e-4),
    RunConfig(arch="mlp", dropout_rate=0.3, l2_strength=1e-4),

    # CNN variants
    RunConfig(arch="cnn", dropout_rate=0.0, l2_strength=0.0),
    RunConfig(arch="cnn", dropout_rate=0.3, l2_strength=0.0),
    RunConfig(arch="cnn", dropout_rate=0.0, l2_strength=1e-4),
    RunConfig(arch="cnn", dropout_rate=0.3, l2_strength=1e-4),
]

for cfg in experiment_configs:
    print(cfg.run_name())

mlp_baseline
mlp_drop0.3
mlp_l20.0001
mlp_drop0.3_l20.0001
cnn_baseline
cnn_drop0.3
cnn_l20.0001
cnn_drop0.3_l20.0001


In [30]:
for cfg in experiment_configs:
    print(f"Running: {cfg.run_name()}")

    model, history, run_info = train_and_evaluate(cfg)

    # Save plots
    save_history_plots(history, cfg.run_name(), FIG_DIR)

    # Save model
    model_path = MODEL_DIR / f"{cfg.run_name()}.keras"
    model.save(model_path)

    # Save metrics JSON
    log_path = LOG_DIR / f"{cfg.run_name()}_metrics.json"
    with open(log_path, "w") as f:
        json.dump(run_info, f, indent=2)

    runs.append(run_info)

    print(
        f"# test_acc={run_info["test_accuracy"]:.4f} "
        f"gap={run_info["generalisation_gap"]:.4f} "
        f"time={run_info["train_time_sec"]:.1f}s "
        f"params={run_info["num_params"]}"
    )

print("Done")

Running: mlp_baseline
# test_acc=0.8677 gap=0.0508 time=96.2s params=101770
Running: mlp_drop0.3
# test_acc=0.8820 gap=0.0124 time=93.7s params=101770
Running: mlp_l20.0001
# test_acc=0.8657 gap=0.0423 time=95.9s params=101770
Running: mlp_drop0.3_l20.0001
# test_acc=0.8785 gap=0.0082 time=86.1s params=101770
Running: cnn_baseline
# test_acc=0.9159 gap=0.0660 time=720.9s params=421642
Running: cnn_drop0.3
# test_acc=0.9178 gap=0.0444 time=704.5s params=421642
Running: cnn_l20.0001
# test_acc=0.9045 gap=0.0638 time=719.1s params=421642
Running: cnn_drop0.3_l20.0001
# test_acc=0.9165 gap=0.0370 time=567.1s params=421642
Done


#### Build a results table (sorted)

In [32]:
# Convert list of dicts to table-like display
def make_summary_table(run_infos):
    # Select and order columns
    rows = []
    for r in run_infos:
        rows.append({
            "run_name": r["run_name"],
            "arch": r["arch"],
            "dropout": r["dropout_rate"],
            "l2": r["l2_strength"],
            "test_acc": r["test_accuracy"],
            "val_acc": r["final_val_accuracy"],
            "train_acc": r["final_train_accuracy"],
            "gap": r["generalisation_gap"],
            "time_sec": r["train_time_sec"],
            "params": r["num_params"],
        })
    # Sort: best test accuracy first
    rows = sorted(rows, key=lambda x: x["test_acc"], reverse=True)
    return rows

summary_rows = make_summary_table(runs)

# Print nicely 
for row in summary_rows:
    print(
        f"{row['run_name']:<22} | "
        f"test={row['test_acc']:.4f} | val={row['val_acc']:.4f} | "
        f"gap={row['gap']:.4f} | time={row['time_sec']:.1f}s | params={row['params']}"
    )

cnn_drop0.3            | test=0.9178 | val=0.9224 | gap=0.0444 | time=704.5s | params=421642
cnn_drop0.3_l20.0001   | test=0.9165 | val=0.9180 | gap=0.0370 | time=567.1s | params=421642
cnn_baseline           | test=0.9159 | val=0.9198 | gap=0.0660 | time=720.9s | params=421642
cnn_l20.0001           | test=0.9045 | val=0.9082 | gap=0.0638 | time=719.1s | params=421642
mlp_drop0.3            | test=0.8820 | val=0.8902 | gap=0.0124 | time=93.7s | params=101770
mlp_drop0.3_l20.0001   | test=0.8785 | val=0.8843 | gap=0.0082 | time=86.1s | params=101770
mlp_baseline           | test=0.8677 | val=0.8780 | gap=0.0508 | time=96.2s | params=101770
mlp_l20.0001           | test=0.8657 | val=0.8723 | gap=0.0423 | time=95.9s | params=101770


In [33]:
# Ensure output directory for tables exists
csv_path = OUTPUT_DIR / "tables"
csv_path.mkdir(parents=True, exist_ok=True)

out_file = csv_path / "regularisation_summary.csv"

# Export aggregated experiment results for reporting / comparison
with open(out_file, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

print(f"Saved: at {out_file}")

Saved: at outputs\tables\regularisation_summary.csv
